## System checking

In [29]:
import sys
print(sys.version)
print(sys.executable)

3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
d:\anaconda\envs\satquery\python.exe


In [30]:
import torch
import transformers
import peft
import timm
import numpy
import bitsandbytes as bnb
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Timm:", timm.__version__)
print("NumPy:", numpy.__version__)
print("bitsandbytes:", bnb.__version__)

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Transformers: 4.37.2
PEFT: 0.10.0
Timm: 0.9.12
NumPy: 1.26.4
bitsandbytes: 0.50.2


In [31]:
import torchvision
print("torchvision",torchvision.__version__)

torchvision 0.26.0+cu128


In [32]:
import sys

internvl_path = r"D:\projects\SIH_26167\SatQuery_AI\InternVL"

if internvl_path not in sys.path:
    sys.path.insert(0, internvl_path)

print(sys.path[0])

D:\projects\SIH_26167\SatQuery_AI\InternVL


In [33]:
import sys

chat_path = r"D:\projects\SIH_26167\SatQuery_AI\InternVL\internvl_chat"

if chat_path not in sys.path:
    sys.path.insert(0, chat_path)

print(sys.path[0])

D:\projects\SIH_26167\SatQuery_AI\InternVL


In [34]:
import internvl

print("InternVL package found!")

InternVL package found!


In [35]:
from internvl.model.internvl_chat import InternVLChatModel

print("InternVLChatModel imported successfully! ✅")

InternVLChatModel imported successfully! ✅


In [36]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from internvl.model.internvl_chat import InternVLChatModel

model_path = "OpenGVLab/InternVL2_5-2B"

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Loading InternVL 2.5-2B...")

Loading InternVL 2.5-2B...


In [37]:
import gc
import torch
from transformers import BitsAndBytesConfig

if "model" in globals():
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gpu_total_gib = torch.cuda.get_device_properties(0).total_memory // (1024 ** 3)
gpu_budget_gib = max(2, gpu_total_gib - 2)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = InternVLChatModel.from_pretrained(
    model_path,
    quantization_config=quant_config,
    device_map="auto",
    max_memory={0: f"{gpu_budget_gib}GiB", "cpu": "24GiB"},
    offload_folder=r"D:\projects\SIH_26167\model_offload",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).eval()

print("Model loaded with 4-bit GPU + CPU offload")
print("GPU budget:", f"{gpu_budget_gib} GiB / {gpu_total_gib} GiB")
print("GPU allocated:", f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Model loaded with 4-bit GPU + CPU offload
GPU budget: 3 GiB / 5 GiB
GPU allocated: 1.98 GB


In [38]:
from transformers import AutoTokenizer

model_path = "OpenGVLab/InternVL2_5-2B"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    use_fast=False
)

print("Tokenizer loaded ✅")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer loaded ✅


In [100]:
image_path = r"D:\projects\SIH_26167\photo2.jpg"

In [150]:
from PIL import Image
from torchvision import transforms
import torch
import inspect

image_path = r"D:\projects\SIH_26167\photo10.jpg"

image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

pixel_values = transform(image).unsqueeze(0).to(
    device="cuda",
    dtype=torch.float16
)

print("Image tensor:", pixel_values.shape)
print("Device:", pixel_values.device)

Image tensor: torch.Size([1, 3, 448, 448])
Device: cuda:0


In [151]:
question = "how many people are in this image?"

generation_config = dict(
    num_beams=1,
    max_new_tokens=100,
    do_sample=False,
)

response = model.chat(
    tokenizer,
    pixel_values,
    question,
    generation_config
)

print("Question:", question)
print("Answer:", response)
print(inspect.getsource(model.chat))

Question: how many people are in this image?
Answer: There are six people in this image.
    def chat(self, tokenizer, pixel_values, question, generation_config, history=None, return_history=False,
             num_patches_list=None, IMG_START_TOKEN='<img>', IMG_END_TOKEN='</img>', IMG_CONTEXT_TOKEN='<IMG_CONTEXT>',
             verbose=False):

        if history is None and pixel_values is not None and '<image>' not in question:
            question = '<image>\n' + question

        if num_patches_list is None:
            num_patches_list = [pixel_values.shape[0]] if pixel_values is not None else []
        assert pixel_values is None or len(pixel_values) == sum(num_patches_list)

        img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
        self.img_context_token_id = img_context_token_id

        template = get_conv_template(self.template)
        template.system_message = self.system_message
        eos_token_id = tokenizer.convert_tokens_to_ids(templa

In [152]:
from internvl.conversation import get_conv_template

In [153]:
import torch
from internvl.conversation import get_conv_template


def get_confidence(model, tokenizer, pixel_values, question, generation_config):
    if "<image>" not in question:
        question = "<image>\n" + question

    IMG_START_TOKEN = "<img>"
    IMG_END_TOKEN = "</img>"
    IMG_CONTEXT_TOKEN = "<IMG_CONTEXT>"
    num_patches = pixel_values.shape[0]

    model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    template = get_conv_template(model.template)
    template.system_message = model.system_message
    eos_token_id = tokenizer.convert_tokens_to_ids(template.sep.strip())
    template.append_message(template.roles[0], question)
    template.append_message(template.roles[1], None)
    query = template.get_prompt()

    image_tokens = (
        IMG_START_TOKEN
        + IMG_CONTEXT_TOKEN * model.num_image_token * num_patches
        + IMG_END_TOKEN
    )
    query = query.replace("<image>", image_tokens, 1)

    model_inputs = tokenizer(query, return_tensors="pt")
    device = model.language_model.device
    input_ids = model_inputs["input_ids"].to(device)
    attention_mask = model_inputs["attention_mask"].to(device)

    config = generation_config.copy()
    config.update({
        "eos_token_id": eos_token_id,
        "output_scores": True,
        "return_dict_in_generate": True,
    })

    with torch.no_grad():
        output = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            **config,
        )

    response = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    response = response.split(template.sep.strip())[0].strip()

    # Align scores with the final generated tokens. InternVL may prepend BOS
    # or return the prompt, so taking the suffix handles both formats.
    sequence = output.sequences[0]
    num_scored_tokens = len(output.scores)
    generated_tokens = sequence[-num_scored_tokens:] if num_scored_tokens else sequence

    token_confidences = []
    for step, score in enumerate(output.scores):
        if step >= len(generated_tokens):
            break
        probabilities = torch.softmax(score.float(), dim=-1)
        token_id = generated_tokens[step].to(score.device)
        token_confidences.append(probabilities[0, token_id].item())

    confidence = (
        sum(token_confidences) / len(token_confidences)
        if token_confidences else 0.0
    )

    print("Generated tokens:", len(generated_tokens))
    print("Scored tokens:", len(token_confidences))
    return response, confidence, token_confidences

In [154]:
response, confidence, token_confidences = get_confidence(
    model,
    tokenizer,
    pixel_values,
    question,
    generation_config,
)
print("Question:", question)
print("Answer:", response)
print("Token probabilities:", [round(value, 6) for value in token_confidences])
print("Model Confidence:", f"{confidence * 100:.2f}%")

Generated tokens: 9
Scored tokens: 9
Question: how many people are in this image?
Answer: There are six people in this image.
Token probabilities: [0.992284, 0.999815, 0.416912, 0.999451, 0.842278, 0.793097, 0.999991, 0.999982, 0.999901]
Model Confidence: 89.37%


In [129]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [130]:
import pickle
from pathlib import Path

pkl_path = Path(r"D:\projects\SIH_26167\SatQuery_AI\satquery_inference_config.pkl")

inference_bundle = {
    "model_path": model_path,
    "tokenizer_path": model_path,
    "generation_config": generation_config.copy(),
    "dataset_root": r"D:\projects\SIH_26167\SatQuery_AI\vqav2_internvl",
    "image_size": 448,
    "num_image_token": int(model.num_image_token),
    "quantization": {
        "load_in_4bit": True,
        "compute_dtype": "float16",
        "quant_type": "nf4",
        "device_map": "auto",
        "offload_folder": r"D:\projects\SIH_26167\model_offload",
    },
}

with pkl_path.open("wb") as f:
    pickle.dump(inference_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("PKL saved:", pkl_path)
print("Model:", inference_bundle["model_path"])
print("GPU/CPU offload config saved: yes")

PKL saved: D:\projects\SIH_26167\SatQuery_AI\satquery_inference_config.pkl
Model: OpenGVLab/InternVL2_5-2B
GPU/CPU offload config saved: yes
